## This notebook implements all 5 next actions:
1. **Hybrid Text Features**: TF-IDF Word (1-2 gram) + Char (3-5 gram) + Gensim Doc2Vec (50-dim).
2. **Expanded Domain Rules & Structural Features**: Multi-merchant keyword rules + text length/word/digit metrics.
3. **Optuna Hyperparameter Tuning**: Parameter search for **LightGBM**, **HistGradientBoosting**, and **XGBoost**.
4. **OOF Confusion Matrix & Error Analysis**: Out-of-Fold confusion matrix evaluation.
5. **Optimal Weighted Blending (SLSQP)**: Optimize ensemble weights $[w_1, w_2, w_3]$ on OOF prediction probabilities to maximize F1-Macro.

## 1. Imports & Setup

In [ ]:
import re
import unicodedata
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.ensemble import HistGradientBoostingClassifier
import lightgbm as lgb
import xgboost as xgb
from scipy.sparse import hstack
from scipy.optimize import minimize

print('All modules loaded successfully!')

## 2. Load Data

In [ ]:
DATA_DIR = '/kaggle/input/competitions/aurora-gate-expense-categorization-challenge'
train = pd.read_csv(f'{DATA_DIR}/train.csv')
test  = pd.read_csv(f'{DATA_DIR}/test.csv')
print(f'Train shape: {train.shape}, Test shape: {test.shape}')
train.head()

## 3. Text Preprocessing, Structural Metrics & Expanded Merchant Rules

In [ ]:
def clean_text(s: str) -> str:
    if not isinstance(s, str):
        return ''
    s = unicodedata.normalize('NFKC', s)
    s = s.lower()
    s = re.sub(r'[#*\-_/\\|@&%$]', ' ', s)
    s = re.sub(r'\s+', ' ', s)
    return s.strip()

train['desc_clean'] = train['description'].apply(clean_text)
test['desc_clean']  = test['description'].apply(clean_text)

# Structural metrics
for df in [train, test]:
    df['char_count']  = df['desc_clean'].apply(len)
    df['word_count']  = df['desc_clean'].apply(lambda x: len(x.split()))
    df['digit_count'] = df['description'].apply(lambda s: sum(c.isdigit() for c in str(s)))

MERCHANT_RULES = [
    # Food & Dining
    ('uber eats', 'Food & Dining'), ('doordash', 'Food & Dining'), ('grubhub', 'Food & Dining'),
    ('seamless', 'Food & Dining'), ('postmates', 'Food & Dining'), ('mcdonald', 'Food & Dining'),
    ('starbucks', 'Food & Dining'), ('chipotle', 'Food & Dining'), ('subway', 'Food & Dining'),
    ('burger king', 'Food & Dining'), ('domino', 'Food & Dining'), ('taco bell', 'Food & Dining'),
    # Transportation
    ('lyft', 'Transportation'), ('uber', 'Transportation'), ('shell', 'Transportation'),
    ('chevron', 'Transportation'), ('exxon', 'Transportation'), ('mobil', 'Transportation'),
    ('bp ', 'Transportation'), ('hertz', 'Transportation'), ('avis', 'Transportation'),
    # Subscriptions
    ('netflix', 'Subscriptions'), ('spotify', 'Subscriptions'), ('hulu', 'Subscriptions'),
    ('amazon prime', 'Subscriptions'), ('apple.com bill', 'Subscriptions'), ('disney', 'Subscriptions'),
    ('hbo', 'Subscriptions'), ('youtube', 'Subscriptions'),
    # Groceries
    ('whole foods', 'Groceries'), ('instacart', 'Groceries'), ('trader joe', 'Groceries'),
    ('kroger', 'Groceries'), ('safeway', 'Groceries'), ('aldi', 'Groceries'),
    # Health & Fitness
    ('planet fitness', 'Health & Fitness'), ('cvs pharmacy', 'Health & Fitness'),
    ('walgreens', 'Health & Fitness'), ('rite aid', 'Health & Fitness'), ('equinox', 'Health & Fitness'),
    # Bills & Utilities
    ('at&t', 'Bills & Utilities'), ('verizon', 'Bills & Utilities'), ('comcast', 'Bills & Utilities'),
    ('con edison', 'Bills & Utilities'), ('t-mobile', 'Bills & Utilities'), ('spectrum', 'Bills & Utilities'),
    # Travel
    ('delta', 'Travel'), ('united', 'Travel'), ('american air', 'Travel'),
    ('southwest', 'Travel'), ('airbnb', 'Travel'), ('expedia', 'Travel'), ('marriott', 'Travel'),
    # Shopping
    ('walmart', 'Shopping'), ('target', 'Shopping'), ('best buy', 'Shopping'),
    ('home depot', 'Shopping'), ('lowe', 'Shopping'), ('amazon', 'Shopping')
]
RULE_CATS = sorted(set(c for _, c in MERCHANT_RULES))
RULE_IDX  = {c: i + 1 for i, c in enumerate(RULE_CATS)}

def rule_feature(desc: str) -> int:
    for kw, cat in MERCHANT_RULES:
        if kw in desc:
            return RULE_IDX[cat]
    return 0

DOW_MAP = {'Monday': 0, 'Tuesday': 1, 'Wednesday': 2, 'Thursday': 3, 'Friday': 4, 'Saturday': 5, 'Sunday': 6}

for df in [train, test]:
    df['rule_feat']  = df['desc_clean'].apply(rule_feature)
    df['dow_num']    = df['day_of_week'].map(DOW_MAP).fillna(-1).astype(int)
    df['is_weekend'] = (df['dow_num'] >= 5).astype(int)
    df['dow_sin']    = np.sin(2 * np.pi * df['dow_num'].clip(0) / 7)
    df['dow_cos']    = np.cos(2 * np.pi * df['dow_num'].clip(0) / 7)
    df['log_amount'] = np.log1p(df['amount'])

print('Text cleaning, merchant rules, and structural features completed.')

## 4. Hybrid Text Feature Extraction & Feature Stacking

In [ ]:
# Word TF-IDF
word_vec = TfidfVectorizer(ngram_range=(1, 2), max_features=800, min_df=2, sublinear_tf=True)
X_word_tr = word_vec.fit_transform(train['desc_clean'])
X_word_te = word_vec.transform(test['desc_clean'])

# Char TF-IDF
char_vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), max_features=1200, min_df=2, sublinear_tf=True)
X_char_tr = char_vec.fit_transform(train['desc_clean'])
X_char_te = char_vec.transform(test['desc_clean'])

# Doc2Vec
all_descriptions = pd.concat([train['desc_clean'], test['desc_clean']]).reset_index(drop=True)
tagged_data = [TaggedDocument(words=text.split(), tags=[str(i)]) for i, text in enumerate(all_descriptions)]

d2v_model = Doc2Vec(vector_size=50, window=5, min_count=2, workers=4, epochs=15, seed=42)
d2v_model.build_vocab(tagged_data)
d2v_model.train(tagged_data, total_examples=d2v_model.corpus_count, epochs=d2v_model.epochs)

def get_d2v_vectors(df, model):
    vectors = []
    for text in df['desc_clean']:
        words = text.split()
        vectors.append(model.infer_vector(words))
    return np.array(vectors)

X_d2v_tr = get_d2v_vectors(train, d2v_model)
X_d2v_te = get_d2v_vectors(test, d2v_model)

# Scaled Numeric Features
NUM_COLS = ['amount', 'log_amount', 'dow_num', 'is_weekend', 'dow_sin', 'dow_cos', 'rule_feat', 'char_count', 'word_count', 'digit_count']
scaler = StandardScaler()
X_num_tr = scaler.fit_transform(train[NUM_COLS])
X_num_te = scaler.transform(test[NUM_COLS])

# Combine all features
X_tr_sparse = hstack([X_word_tr, X_char_tr, X_d2v_tr, X_num_tr]).tocsr()
X_te_sparse = hstack([X_word_te, X_char_te, X_d2v_te, X_num_te]).tocsr()

X_train_dense = X_tr_sparse.toarray()
X_test_dense  = X_te_sparse.toarray()

le = LabelEncoder()
y_train = le.fit_transform(train['category'])

print(f'Combined Hybrid Feature Matrix shape: {X_tr_sparse.shape}')
print(f'Target classes ({len(le.classes_)}): {list(le.classes_)}')

## 5. Optuna Hyperparameter Optimization (LightGBM, HistGBM, XGBoost)

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def print_trial_callback(study, trial):
    print(f'  Trial {trial.number+1}/{len(study.trials)} | Val F1-Macro: {trial.value:.4f}')

print('Tuning LightGBM...')
def objective_lgb(trial):
    params = {
        'num_leaves': trial.suggest_int('num_leaves', 31, 63),
        'max_depth': trial.suggest_int('max_depth', 5, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.04, 0.10, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 150, 250, step=50),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 1.0, log=True),
        'class_weight': 'balanced',
        'random_state': 42,
        'n_jobs': 4,
        'verbose': -1
    }
    oof_preds = np.zeros(len(train))
    for tr_idx, val_idx in skf.split(X_tr_sparse, y_train):
        m = lgb.LGBMClassifier(**params)
        m.fit(X_tr_sparse[tr_idx], y_train[tr_idx])
        oof_preds[val_idx] = m.predict(X_tr_sparse[val_idx])
    return f1_score(y_train, oof_preds, average='macro')

study_lgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_lgb.optimize(objective_lgb, n_trials=5, callbacks=[print_trial_callback])
print(f'-> Best LightGBM F1-Macro: {study_lgb.best_value:.4f}')

print('\nTuning HistGradientBoosting...')
def objective_hgb(trial):
    params = {
        'max_iter': trial.suggest_int('max_iter', 150, 250, step=50),
        'max_leaf_nodes': trial.suggest_int('max_leaf_nodes', 31, 63),
        'learning_rate': trial.suggest_float('learning_rate', 0.04, 0.10, log=True),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 10, 30),
        'l2_regularization': trial.suggest_float('l2_regularization', 1e-4, 1.0, log=True),
        'class_weight': 'balanced',
        'random_state': 42
    }
    oof_preds = np.zeros(len(train))
    for tr_idx, val_idx in skf.split(X_train_dense, y_train):
        m = HistGradientBoostingClassifier(**params)
        m.fit(X_train_dense[tr_idx], y_train[tr_idx])
        oof_preds[val_idx] = m.predict(X_train_dense[val_idx])
    return f1_score(y_train, oof_preds, average='macro')

study_hgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_hgb.optimize(objective_hgb, n_trials=5, callbacks=[print_trial_callback])
print(f'-> Best HistGBM F1-Macro: {study_hgb.best_value:.4f}')

print('\nTuning XGBoost...')
def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 150, 250, step=50),
        'max_depth': trial.suggest_int('max_depth', 4, 7),
        'learning_rate': trial.suggest_float('learning_rate', 0.04, 0.10, log=True),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'gamma': trial.suggest_float('gamma', 0.0, 0.5),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 1.0, log=True),
        'random_state': 42,
        'n_jobs': 4,
        'eval_metric': 'mlogloss'
    }
    oof_preds = np.zeros(len(train))
    for tr_idx, val_idx in skf.split(X_tr_sparse, y_train):
        m = xgb.XGBClassifier(**params)
        m.fit(X_tr_sparse[tr_idx], y_train[tr_idx])
        oof_preds[val_idx] = m.predict(X_tr_sparse[val_idx])
    return f1_score(y_train, oof_preds, average='macro')

study_xgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_xgb.optimize(objective_xgb, n_trials=5, callbacks=[print_trial_callback])
print(f'-> Best XGBoost F1-Macro: {study_xgb.best_value:.4f}')

## 6. OOF Prediction Probability Extraction & Confusion Matrix

In [ ]:
best_lgb_params = study_lgb.best_params.copy()
best_lgb_params.update({'class_weight': 'balanced', 'random_state': 42, 'n_jobs': 4, 'verbose': -1})

best_hgb_params = study_hgb.best_params.copy()
best_hgb_params.update({'class_weight': 'balanced', 'random_state': 42})

best_xgb_params = study_xgb.best_params.copy()
best_xgb_params.update({'random_state': 42, 'n_jobs': 4, 'eval_metric': 'mlogloss'})

oof_probs_lgb = np.zeros((len(train), len(le.classes_)))
oof_probs_hgb = np.zeros((len(train), len(le.classes_)))
oof_probs_xgb = np.zeros((len(train), len(le.classes_)))

test_probs_lgb = np.zeros((len(test), len(le.classes_)))
test_probs_hgb = np.zeros((len(test), len(le.classes_)))
test_probs_xgb = np.zeros((len(test), len(le.classes_)))

for tr_idx, val_idx in skf.split(X_tr_sparse, y_train):
    m_lgb = lgb.LGBMClassifier(**best_lgb_params)
    m_lgb.fit(X_tr_sparse[tr_idx], y_train[tr_idx])
    oof_probs_lgb[val_idx] = m_lgb.predict_proba(X_tr_sparse[val_idx])
    test_probs_lgb += m_lgb.predict_proba(X_te_sparse) / 5.0
    
    m_hgb = HistGradientBoostingClassifier(**best_hgb_params)
    m_hgb.fit(X_train_dense[tr_idx], y_train[tr_idx])
    oof_probs_hgb[val_idx] = m_hgb.predict_proba(X_train_dense[val_idx])
    test_probs_hgb += m_hgb.predict_proba(X_test_dense) / 5.0
    
    m_xgb = xgb.XGBClassifier(**best_xgb_params)
    m_xgb.fit(X_tr_sparse[tr_idx], y_train[tr_idx])
    oof_probs_xgb[val_idx] = m_xgb.predict_proba(X_tr_sparse[val_idx])
    test_probs_xgb += m_xgb.predict_proba(X_te_sparse) / 5.0

lgbm_f1_macro = f1_score(y_train, np.argmax(oof_probs_lgb, axis=1), average="macro")
hgb_f1_macro = f1_score(y_train, np.argmax(oof_probs_hgb, axis=1), average="macro")
xgb_f1_macro = f1_score(y_train, np.argmax(oof_probs_xgb, axis=1), average="macro")

print(f'LGBM OOF F1-Macro: {lgbm_f1_macro}')
print(f'HGB  OOF F1-Macro: {hgb_f1_macro}')
print(f'XGB  OOF F1-Macro: {xgb_f1_macro}')

B = lgbm_f1_macro+hgb_f1_macro+xgb_f1_macro



cm = confusion_matrix(y_train, np.argmax(oof_probs_lgb, axis=1))
pd.DataFrame(cm, index=le.classes_, columns=le.classes_)

## 7. Optimal Weighted Blending (SLSQP Optimization)

In [ ]:
def loss_func(weights):
    w1, w2, w3 = weights
    blend_oof = w1 * oof_probs_lgb + w2 * oof_probs_hgb + w3 * oof_probs_xgb
    preds = np.argmax(blend_oof, axis=1)
    return -f1_score(y_train, preds, average='macro')

init_weights = [lgbm_f1_macro/B,hgb_f1_macro/B , xgb_f1_macro/B]
bounds = [(0, 1), (0, 1), (0, 1)]
constraints = ({'type': 'eq', 'fun': lambda w: 1.0 - sum(w)})

res = minimize(loss_func, init_weights, method='SLSQP', bounds=bounds, constraints=constraints)
w1, w2, w3 = res.x
best_f1 = -res.fun

print(f'Optimal Ensemble Weights -> LightGBM: {w1:.3f}, HistGBM: {w2:.3f}, XGBoost: {w3:.3f}')
print(f'✨ Final Weighted Ensemble OOF F1-Macro: {best_f1:.4f}')

## 8. Generate Submission File

In [ ]:
final_test_probs = w1 * test_probs_lgb + w2 * test_probs_hgb + w3 * test_probs_xgb
final_preds_idx  = np.argmax(final_test_probs, axis=1)
final_preds      = le.inverse_transform(final_preds_idx)
final_preds


In [ ]:
sub = pd.DataFrame({
    'transaction_id': test['transaction_id'],
    'category':       final_preds,
})
sub.to_csv('submission.csv', index=False)
sub.head(10)